In [ ]:
# Find this checkout from a notebook or a kernel launched at the workspace root.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents, _here / 'cic17-cic18-transfer-study']
if os.environ.get('NIDS_STUDY_ROOT'):
    _candidates.insert(0, Path(os.environ['NIDS_STUDY_ROOT']).expanduser().resolve())
REPO_ROOT = next((p for p in _candidates if (p / 'src/nids_transfer').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this checkout or set NIDS_STUDY_ROOT.')
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
from nids_transfer.paths import study_paths
DATA_DIR, ARTIFACT_DIR = study_paths(REPO_ROOT)
print('Data:', DATA_DIR)
print('Artifacts:', ARTIFACT_DIR)


# Dataset compatibility: CIC2017 and CIC2018

Keep only exact shared column names, in 2017 order. Use the 2017 Parquet types as the reference for 2018. Source files are preserved. Scaling, feature selection, and protocol encoding come later.

Run with a kernel containing `duckdb` (`%pip install duckdb` if needed). DuckDB processes the Parquet files without loading both datasets into pandas.

In [ ]:
from pathlib import Path
import duckdb

ROOT = REPO_ROOT
if not (DATA_DIR / "CIC2017_improved.parquet").exists():
    raise FileNotFoundError("Set NIDS_DATA_DIR to the directory containing the input Parquet files.")
PATH_2017 = DATA_DIR / "CIC2017_improved.parquet"
PATH_2018 = DATA_DIR / "CIC2018_improved_sample.parquet"
OUT = DATA_DIR / "compatible"
con = duckdb.connect()

def ident(name):
    return '"' + name.replace('"', '""') + '"'

def literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def schema(path):
    return dict((row[0], row[1]) for row in con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]
    ).fetchall())

s17, s18 = schema(PATH_2017), schema(PATH_2018)
shared = [col for col in s17 if col in s18]
assert shared, "No shared columns found"
print("Dropped from 2017:", sorted(s17.keys() - s18.keys()))
print("Dropped from 2018:", sorted(s18.keys() - s17.keys()))
print("Shared columns:", len(shared))
for col in shared:
    if s17[col] != s18[col]:
        print(f"{col}: {s18[col]} -> {s17[col]}")

## Check conversions

Count non-null 2018 values that cannot be cast. If any exist, stop and inspect examples; do not silently replace them with missing values. Existing nulls remain nulls. Casts follow DuckDB conversion rules, which can round fractional values when casting to integer types.

In [ ]:
source18 = f"read_parquet({literal(PATH_2018)})"
checks = [
    f"count(*) FILTER (WHERE {ident(col)} IS NOT NULL "
    f"AND TRY_CAST({ident(col)} AS {s17[col]}) IS NULL) AS {ident(col)}"
    for col in shared
]
counts = con.execute(f"SELECT {', '.join(checks)} FROM {source18}").fetchone()
failures = {col: count for col, count in zip(shared, counts) if count}
for col, count in failures.items():
    examples = con.execute(
        f"SELECT DISTINCT {ident(col)} FROM {source18} "
        f"WHERE {ident(col)} IS NOT NULL AND TRY_CAST({ident(col)} AS {s17[col]}) IS NULL LIMIT 5"
    ).fetchall()
    print(col, "failed values:", count, "examples:", examples)
if failures:
    raise ValueError("Resolve the conversion failures above before exporting.")
print("All non-null 2018 values can be cast to the corresponding 2017 types.")

## Save compatible copies

Both outputs contain the same columns, in the same order, with identical types. No rows are filtered. Existing output files cause an error to prevent accidental replacement.

In [ ]:
assert not failures, "Resolve conversion failures first"
OUT.mkdir(parents=True, exist_ok=True)
out17 = OUT / "CIC2017_shared.parquet"
out18 = OUT / "CIC2018_shared.parquet"
for path in (out17, out18):
    if path.exists():
        raise FileExistsError(f"Output already exists: {path}")

select17 = ', '.join(ident(col) for col in shared)
select18 = ', '.join(f"CAST({ident(col)} AS {s17[col]}) AS {ident(col)}" for col in shared)
for source, selection, target in [(PATH_2017, select17, out17), (PATH_2018, select18, out18)]:
    con.execute(
        f"COPY (SELECT {selection} FROM read_parquet({literal(source)})) "
        f"TO {literal(target)} (FORMAT PARQUET, COMPRESSION ZSTD)"
    )

expected = [(col, s17[col]) for col in shared]
assert list(schema(out17).items()) == expected
assert list(schema(out18).items()) == expected
for source, target in [(PATH_2017, out17), (PATH_2018, out18)]:
    original_count = con.execute("SELECT count(*) FROM read_parquet(?)", [str(source)]).fetchone()[0]
    output_count = con.execute("SELECT count(*) FROM read_parquet(?)", [str(target)]).fetchone()[0]
    assert original_count == output_count
    print(f"{target.name}: {output_count:,} rows; {len(shared)} columns")
print("Compatibility verified: column order, types, and row counts.")